# HubbardBath 05 — Jordan–Wigner: fermions to qubits

This notebook starts the quantum-algorithm half of HubbardBath. We take the same validated two-site Fermi–Hubbard Hamiltonian and rewrite it as a weighted sum of Pauli strings on four qubits.

**Goal:** derive and verify

$$H=\sum_j \alpha_j P_j$$

with no loss of information relative to the fermionic Hamiltonian.


In [ ]:
from pathlib import Path
import sys
import numpy as np

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'HubbardBath']
for candidate in candidates:
    if (candidate / 'src' / 'qubit_mapping.py').exists():
        repo_root = candidate.resolve()
        break
else:
    raise FileNotFoundError('Run this notebook from the HubbardBath repository.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.hubbard import two_site_hubbard_hamiltonian
from src.qubit_mapping import (
    dense_pauli_decomposition,
    lcu_normalization,
    reconstruct_from_pauli_terms,
    remove_zero_terms,
    two_site_hubbard_pauli_terms,
)


## 1. Qubit ordering

We use four qubits, one per spin-orbital:

```text
q0 = site 0 ↑
q1 = site 0 ↓
q2 = site 1 ↑
q3 = site 1 ↓
```

Jordan–Wigner represents the fermionic annihilation operator as

$$c_j = Z_0 Z_1 \cdots Z_{j-1}\,\sigma^-_j.$$

The preceding `Z` string is the fermionic parity bookkeeping.


## 2. Analytic Pauli decomposition

For the two-site model, the interaction terms become `Z` and `ZZ` operators while hopping produces `XZX` and `YZY` parity strings.


In [ ]:
t, U, mu = 1.0, 4.0, 0.0
terms = remove_zero_terms(two_site_hubbard_pauli_terms(t=t, U=U, mu=mu))

for label, coefficient in sorted(terms.items()):
    print(f'{coefficient:+.3f}  {label}')

print('\nNumber of non-zero Pauli terms:', len(terms))


## 3. Verify it is exactly the same Hamiltonian

We reconstruct the dense qubit Hamiltonian from the Pauli sum and compare it directly with the fermionic Hamiltonian used in Milestones 01–04.


In [ ]:
H_fermion = two_site_hubbard_hamiltonian(t=t, U=U, mu=mu)
H_qubit = reconstruct_from_pauli_terms(terms)

matrix_error = np.linalg.norm(H_qubit - H_fermion, ord=2)
spectrum_error = np.max(np.abs(
    np.linalg.eigvalsh(H_qubit) - np.linalg.eigvalsh(H_fermion)
))

print(f'Operator-norm matrix error: {matrix_error:.3e}')
print(f'Maximum spectrum error:    {spectrum_error:.3e}')

assert matrix_error < 1e-12
assert spectrum_error < 1e-12


## 4. Independent Pauli-basis recovery

To avoid validating one analytic formula with itself, we also recover every Pauli coefficient directly from the dense Hamiltonian using

$$\alpha_P=\frac{\mathrm{Tr}(P^\dagger H)}{2^n}.$$


In [ ]:
numeric_terms = dense_pauli_decomposition(H_fermion)

print('Analytic and numerical term sets identical:', terms.keys() == numeric_terms.keys())
for label in sorted(terms):
    print(label, terms[label], numeric_terms[label])

assert terms.keys() == numeric_terms.keys()
for label in terms:
    assert abs(terms[label] - numeric_terms[label]) < 1e-12


## 5. The quantity we will reuse for block encoding

For an LCU decomposition, define

$$\alpha=\sum_j|\alpha_j|.$$

Later we will normalize the Hamiltonian as `H/α` inside a larger unitary block encoding.


In [ ]:
alpha = lcu_normalization(terms)
print(f'LCU normalization alpha = {alpha:.6f}')


## Milestone result

We have now connected the original interacting-fermion model to an explicit qubit Hamiltonian with exact numerical equivalence.

The next milestone uses these Pauli terms to implement and benchmark **Hamiltonian simulation**: exact $e^{-iHt}$ versus product-formula/Trotter approximations.
